# Introduction

In this notebook, I fine-tune the **mistral-7b-instruct-v0.2-bnb-4bit** language model on a synthetic code-repair dataset to improve its ability to automatically detect and correct bugs in Python code for AI/ML-related projects.

The dataset consists of 500+ training samples, where each example provides a task context (title, description, difficulty), a buggy implementation (`incorrect_code`), and the corresponding fixed solution (`correct_code`), often accompanied by an error hint (`error_type`). The objective is to train the model to take the problem context and incorrect code as input and produce the corrected Python code as output—reliably and in a format suitable for real development workflows.

In the following sections, I will:

* Load and validate the JSON dataset

* Convert each sample into an instruction-style prompt/completion format

* Configure the tokenizer and fine-tuning setup (LoRA/QLoRA depending on the training configuration)

* Train the model and monitor learning behavior

* Run inference on new buggy code examples to evaluate the model’s correction quality

# 1. Environment Setup and Dependency Installation

This section prepares the notebook runtime for fine-tuning Mistral v0.3 7B Instruct. I enable an Unsloth performance flag for improved throughput and extended context handling, then install the required libraries for training with Unsloth + vLLM. The dependencies also include the core tooling needed to load and preprocess the JSON dataset and to run parameter-efficient fine-tuning (LoRA/QLoRA) efficiently on a Colab GPU.

In [1]:
%%capture
import os

# Unsloth performance / longer context support
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"  # Enables longer context support in some setups

# Install dependencies (Colab vs non-Colab)
if "COLAB_" not in "".join(os.environ.keys()):
    # Local / non-Colab environment
    !pip install -U unsloth vllm datasets accelerate peft bitsandbytes
    !pip install -U transformers==4.56.2 trl==0.22.2
else:
    # Colab environment (faster + cleaner installs via uv)
    !pip install --upgrade -qqq uv
    !uv pip install -q vllm==0.11.2 unsloth-zoo unsloth
    !uv pip install -q transformers==4.56.2
    !uv pip install -q --no-deps trl==0.22.2
    !uv pip install -q datasets accelerate peft bitsandbytes


# 2. Model Initialization and QLoRA Configuration

Here I load Mistral v0.3 7B Instruct in 4-bit mode and attach LoRA adapters to the model’s attention and MLP projection layers. This QLoRA-style approach significantly reduces memory requirements while allowing effective fine-tuning. Because the notebook runs on an A100 GPU, I increase the maximum sequence length to better accommodate longer code-repair samples.

In [2]:
from unsloth import FastLanguageModel
import torch

# Sequence length controls how much context (prompt + code) the model can see.
max_seq_length = 7680

# LoRA rank controls adapter capacity (higher = more capacity but slower + more VRAM).
lora_rank = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype= None,
    load_in_4bit=True,          # QLoRA-style training (memory efficient)
    fast_inference=False,        # Enables vLLM fast inference when supported
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.5 # Lower this if you hit OOM
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Mistral patching. Transformers: 4.56.2. vLLM: 0.11.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_rank,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2026.1.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


# 3. Data Preparation and Prompt Formatting

In this section, I load the synthetic JSON dataset (`final_dataset.json`) and convert each sample into an instruction-style training example suitable for fine-tuning Mistral v0.3 7B Instruct. Each record is transformed into a structured prompt containing the task context, the buggy code, and the error hint, while the expected completion is the corrected code. I also compute token-length statistics to choose an appropriate maximum sequence length and prevent silent truncation during training.

In [4]:
from datasets import load_dataset

DATA_PATH = "final_dataset.json"
raw_dataset = load_dataset("json", data_files=DATA_PATH, split="train")

print("Loaded samples:", len(raw_dataset))

Generating train split: 0 examples [00:00, ? examples/s]

Loaded samples: 582


In [5]:
SYSTEM_PROMPT = (
    "You are an AI code fixer specialized in Python for AI/ML projects. "
    "Given a task description and buggy code, identify and fix any issues so the program runs correctly "
    "and matches the task requirements. "
    "Sometimes an error report is provided, sometimes it is not. "
    "Output ONLY the corrected Python code."
)

In [9]:
import random

ERROR_HINT_PROB = 0.50
random.seed(42)

def _canonical_error_type(error_type_str: str) -> str:
    # the dataset values look like: "NameError: ... || line: ..."
    # Keep only the leading error class label (e.g., "NameError")
    if not isinstance(error_type_str, str):
        return "Unknown"
    return error_type_str.split(":", 1)[0].strip() or "Unknown"

def to_training_text(ex):
    include_hint = (random.random() < ERROR_HINT_PROB)

    user_prompt = (
        f"Task title: {ex['title']}\n"
        f"Difficulty: {ex['difficulty']}\n"
        f"Task description: {ex['description']}\n\n"
    )

    if include_hint:
        user_prompt += (
            "Bug report (optional hint):\n"
            f"{ex['error_type']}\n\n"
        )

    user_prompt += (
        "Incorrect code:\n"
        "```python\n"
        f"{ex['incorrect_code']}\n"
        "```\n\n"
        "Fix the code and identify the error type you fixed.\n"
        "Return ONLY the tagged output.\n"
    )

    assistant_output = (
        "<correct_code>\n"
        f"{ex['correct_code']}\n"
        "</correct_code>\n"
        "<error_type>\n"
        f"{_canonical_error_type(ex['error_type'])}\n"
        "</error_type>"
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": assistant_output},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}


In [10]:
dataset = raw_dataset.map(to_training_text, remove_columns=raw_dataset.column_names, desc="Formatting dataset")
print(dataset[0]["text"])

Formatting dataset:   0%|          | 0/582 [00:00<?, ? examples/s]

<s>[INST] Task title: Adult Income Hyperparameter Grid
Difficulty: easy
Task description: Fetch the adult dataset via sklearn's fetch_openml. Use GridSearchCV to tune a GradientBoostingClassifier's number of estimators, learning rate, and max depth. Report the best parameters and test accuracy.

Incorrect code:
```python
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# Fetch the adult dataset from OpenML
print("Fetching adult dataset...")
adult = fetch_openml('adult', version=2, as_frame=True, parser='auto')
X = adult.data
y = adult.target

# Handle categorical variables by encoding them
print("Preprocessing data...")
le = LabelEncoder()
for col in X.select_dtypes(include=['category', 'object']).columns:
    X[col] = le.fit_transform(X[col].astype(str))

# Handl

**!NOTE:**

The formatted sample above shows the exact supervised fine-tuning (SFT) structure used for training. Everything inside the [INST] ... [/INST] block represents the input prompt (task context, bug report, and the buggy incorrect_code). The content that appears after [/INST] is the expected assistant response, which is the original correct_code. During training, the model learns to generate this corrected code continuation given the prompt, effectively learning a “buggy code → fixed code” mapping in an instruction-following format compatible with Mistral Instruct.

## 3.1. Token Length Analysis and Sequence-Length Selection

Before starting training, I measure the token length of each formatted example to understand how much context the model must process per sample. This step prevents silent truncation of prompts or corrected code during fine-tuning and helps me choose an appropriate `max_seq_length` that balances training quality, speed, and GPU memory usage.

In [11]:
import numpy as np

def add_token_len(ex):
    ex["n_tokens"] = len(tokenizer(ex["text"]).input_ids)
    return ex

ds_len = dataset.map(add_token_len, desc="Counting tokens")
lengths = np.array(ds_len["n_tokens"])

print("Samples:", len(lengths))
print("Min:", lengths.min())
print("Median:", int(np.median(lengths)))
print("P90:", int(np.percentile(lengths, 90)))
print("P95:", int(np.percentile(lengths, 95)))
print("Max:", lengths.max())


Counting tokens:   0%|          | 0/582 [00:00<?, ? examples/s]

Samples: 582
Min: 1129
Median: 2642
P90: 4577
P95: 5227
Max: 7154


**!CONCLUSION:**

The dataset contains 582 samples with a median length of 2,625 tokens, while 95% of the samples are below 5,191 tokens. The longest example is 7,168 tokens, which indicates that a max_seq_length of 8,192 will safely cover the full dataset without truncation. This confirms that the formatted prompts and corrected-code outputs fit within a long-context training setup and that the data is ready to proceed to the fine-tuning stage.

## 3.2. Train-Validation Split

In this step, I split the formatted dataset into a training set and a validation (evaluation) set. The training split is used to update the model’s LoRA parameters, while the validation split is held out to monitor generalization during fine-tuning. Keeping a separate evaluation set helps detect overfitting and provides an unbiased signal for model selection and hyperparameter tuning.


In [12]:
split = dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print("Train:", len(train_dataset))
print("Eval :", len(eval_dataset))


Train: 494
Eval : 88


**!NOTE:**

With a `test_size=0.15`, the dataset is divided into 494 training samples and 88 evaluation samples. This allocation preserves most data for learning while reserving a meaningful subset for validation, enabling reliable evaluation of how well the fine-tuned model performs on unseen code-fixing examples.

# 4. Supervised Fine-Tuning Configuration and Training

In this stage, I fine-tune the QLoRA-adapted **Mistral v0.3 7B Instruct** model using TRL’s `SFTTrainer`. The trainer learns to generate the corrected Python code (assistant output) given the task context and buggy implementation (user prompt). Since the dataset contains long examples (up to ~7k tokens), I use an 8k sequence length and conservative batch settings to keep training stable on an A100 GPU.

In [13]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,  # Keep False for long samples (packing mainly helps short texts)
    args=SFTConfig(
        per_device_train_batch_size=1,        # long context => reduce batch size
        gradient_accumulation_steps=8,        # effective batch size ~8
        warmup_steps=10,
        num_train_epochs=1,                   # recommended over max_steps for real training
        learning_rate=2e-4,                   # good starting point for QLoRA
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=10,                        # adjust based on speed
        save_strategy="steps",
        save_steps=50,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/494 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/88 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [14]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-80GB. Max memory = 79.318 GB.
6.914 GB of memory reserved.


In [15]:
trainer_stats = trainer.train()
trainer_stats

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 494 | Num Epochs = 1 | Total steps = 62
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
10,0.288600,0.238901
20,0.214100,0.181283
30,0.165800,0.163090
40,0.150300,0.151798
50,0.159300,0.146614
60,0.145000,0.143394


Unsloth: Not an error, but MistralForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


TrainOutput(global_step=62, training_loss=0.194837381762843, metrics={'train_runtime': 583.191, 'train_samples_per_second': 0.847, 'train_steps_per_second': 0.106, 'total_flos': 6.08770754925527e+16, 'train_loss': 0.194837381762843, 'epoch': 1.0})

# 5. Inference and Qualitative Evaluation

After fine-tuning, I switch the model to inference mode and test it on unseen buggy code examples. The goal is to verify that the model produces only corrected Python code, fixes the intended error, and maintains the task requirements. I also run a small qualitative evaluation on the validation split by comparing the generated output with the expected `correct_code`.

In [16]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32768, 4096, padding_idx=770)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj):

In [32]:
SYSTEM_PROMPT = (
    "You are an AI code fixer specialized in Python for AI/ML projects. "
    "Fix the provided Python code to satisfy the task description and run without errors. "
    "Prefer minimal edits and keep the original structure. "
    "If multiple fixes are possible, choose the safest and most robust one. "
    "Output ONLY the corrected Python code (no explanations, no markdown)."
)


In [35]:
def build_user_prompt(title, description, difficulty, incorrect_code, error_type=None):
    p = (
        f"Task title: {title}\n"
        f"Difficulty: {difficulty}\n"
        f"Task description: {description}\n\n"
    )
    if error_type is not None:
        p += f"Bug report (optional hint): {_canonical_error_type(error_type)}\n\n"
    p += (
        "Incorrect code:\n"
        "```python\n"
        f"{incorrect_code}\n"
        "```\n\n"
        "Find and fix any bugs. Output ONLY the corrected Python code.\n"
    )
    return p


In [19]:
def _clean_code(text: str) -> str:
    text = text.strip()
    # remove markdown fences if the model adds them
    if text.startswith("```"):
        text = text.strip("`").strip()
        if text.lower().startswith("python"):
            text = text[len("python"):].lstrip()
    return text.strip()


In [23]:
import torch

@torch.inference_mode()
def generate_fix(
    title, description, difficulty, incorrect_code,
    error_type=None,               # set None for “realistic” mode
    max_new_tokens=3000,
    temperature=0.0,
):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(title, description, difficulty, incorrect_code, error_type)},
    ]

    # Build prompt string with the correct chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Tokenize to get BOTH input_ids and attention_mask (prevents pad/eos warning)
    enc = tokenizer(prompt, return_tensors="pt")
    input_ids = enc["input_ids"].to(model.device)
    attention_mask = enc["attention_mask"].to(model.device)

    # Deterministic decode for code repair
    out = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0.0),
        temperature=temperature,
        use_cache=True,
    )

    gen_ids = out[0][input_ids.shape[1]:]
    return _clean_code(tokenizer.decode(gen_ids, skip_special_tokens=True))

In [38]:
i = 228
ex = raw_dataset[i]

pred = generate_fix(
    title=ex["title"],
    description=ex["description"],
    difficulty=ex["difficulty"],
    incorrect_code=ex["incorrect_code"],
    error_type=ex["error_type"],
)

print("=== Predicted fix ===\n")
print(pred[:1500])
print("\n=== Expected fix (start) ===\n")
print(ex["correct_code"][:1500])


=== Predicted fix ===

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Since seaborn's penguins dataset requires internet access,
# we'll create a synthetic penguins-like dataset with similar structure
# The penguins dataset typically has: bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g, species

def create_synthetic_penguins_data(n_samples=300):
    """
    Create a synthetic dataset mimicking the Palmer Penguins dataset.
    Three species: Adelie, Chinstrap, Gentoo
    Features: bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g
    """
    # Define species
    species_list = ['Adelie', 'Chinstrap', 'Gentoo']
    n_per_species = n_samples // 3
    
    # Generate data for each species with different characteristics
    data = []
    
    # Adelie penguins (smaller, shorter bills)
    adelie_bill_length = np.random.normal(38.8, 2.7, n_per_species)
    adel

In [39]:
# Checking the predicted answer with true answer
pred_norm = pred.strip()
gold_norm = ex["correct_code"].strip()
print("Exact match:", pred_norm == gold_norm)


Exact match: False


In [30]:
import difflib

def show_diff(pred_code: str, gold_code: str, context_lines: int = 3):
    pred_lines = pred_code.splitlines(keepends=True)
    gold_lines = gold_code.splitlines(keepends=True)

    diff = difflib.unified_diff(
        gold_lines, pred_lines,
        fromfile="expected_correct_code",
        tofile="predicted_fix",
        n=context_lines
    )
    print("".join(diff))


In [31]:
show_diff(pred_norm, gold_norm, context_lines=5)

--- expected_correct_code
+++ predicted_fix
@@ -19,11 +19,11 @@
 reverse_word_index = {value: key for key, value in word_index.items()}
 
 # Decode sequences back to text
 # Note: indices are offset by 3 because 0, 1, 2 are reserved for padding, start, unknown
 def decode_review(encoded_review):
-    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])
+    return ' '.join([reverse_word_index[i - 3] for i in encoded_review])
 
 # Convert all training and test sequences to text
 train_texts = [decode_review(seq) for seq in x_train]
 test_texts = [decode_review(seq) for seq in x_test]
 



In [24]:
import random

random.seed(42)

for i in random.sample(range(len(raw_dataset)), 5):
    ex = raw_dataset[i]

    pred = generate_fix(
        title=ex["title"],
        description=ex["description"],
        difficulty=ex["difficulty"],
        incorrect_code=ex["incorrect_code"],
        error_type=None,
        max_new_tokens=2000,
        temperature=0.0,
    )

    pred_norm = pred.strip()
    gold_norm = ex["correct_code"].strip()

    print("=" * 90)
    print(f"Index: {i} | Title: {ex['title']} | Difficulty: {ex['difficulty']}")
    print("Exact match:", pred_norm == gold_norm)

    # If not exact, show a small diff-friendly preview
    if pred_norm != gold_norm:
        print("\n--- Pred (start) ---")
        print(pred_norm[:400])
        print("\n--- Gold (start) ---")
        print(gold_norm[:400])


Index: 114 | Title: Digits Logistic Regression Baseline | Difficulty: easy
Exact match: True
Index: 25 | Title: Breast Cancer Logistic Regression Scaling | Difficulty: easy
Exact match: True
Index: 281 | Title: Titanic Cross Validation Dashboard | Difficulty: easy
Exact match: True
Index: 250 | Title: Reuters Topic Classification Multinomial NB | Difficulty: easy
Exact match: False

--- Pred (start) ---
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Load the Reuters dataset from keras.datasets
# 

--- Gold (start) ---
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes impor

# 6. Export the Fine-Tuned Model as GGUF for Offline Use

In this step, I export the fine-tuned model to the **GGUF** format, which is compatible with llama.cpp-based runtimes and tools such as **Ollama**. During this export, the learned LoRA updates are merged into the base model, producing a single, self-contained model file that can be used offline without requiring the separate adapter weights. I use the `q4_k_m` quantization method (4-bit) because it offers an excellent balance between model quality, inference speed, and disk/memory footprint for local deployment.

In [40]:
GGUF_DIR = "mistral_codefix_gguf_q4_k_m"
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00003.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Unsloth: Preparing safetensor model files:  67%|██████▋   | 2/3 [00:23<00:11, 11.89s/it]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 3/3 [01:01<00:00, 20.57s/it]


Unsloth: Merge process complete. Saved to `/content/mistral_codefix_gguf_q4_k_m`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['mistral-7b-instruct-

{'save_directory': 'mistral_codefix_gguf_q4_k_m',
 'gguf_files': ['mistral-7b-instruct-v0.3.Q4_K_M.gguf'],
 'modelfile_location': '/content/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [46]:
from google.colab import files
files.download("mistral-7b-instruct-v0.3.Q4_K_M.gguf")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [47]:
from google.colab import drive
drive.mount("/content/drive")
!cp "mistral-7b-instruct-v0.3.Q4_K_M.gguf" "/content/drive/MyDrive/"


Mounted at /content/drive
